In [ ]:
# ! pip install google-genai

In [4]:
%load_ext autoreload
%autoreload 2
import dotenv
dotenv.load_dotenv()
from store_front_generation import *

place_list = {
    "restaurant": 101,
    "fast_food": 40,
    "cafe": 28,
    "bank": 19,
    "bicycle_rental": 10,
    "pub": 9,
    "parking_entrance": 7,
    "clinic": 4,
    "pharmacy": 4,
    "atm": 4,
    "university": 2,
    "bureau_de_change": 2,
    "post_office": 2,
    "theatre": 2,
    "place_of_worship": 1,
    "community_centre": 1,
    "food_court": 1,
    "car_rental": 1,
    "toilets": 1
}


place_type = "restaurant"

prompt_rgb = \
"""A highly detailed, photorealistic orthographic front elevation view of a {place_type} storefront facade, intended for use as a flat texture in a simulation.

Layout:
- ABSOLUTELY NO sidewalk, pavement, street furniture, or surrounding building structure is visible.
- The facade occupies the entire frame, WITHOUT ANY borders or outer walls or background.
- The outer left and right edges of the storefront's main structural frame and glazing system align exactly with the left and right borders of the image canvas. No brick wall, siding, or adjacent building material is visible on the sides.
- The door could be on the left, right, or in the middle.

Signage & Branding: If the store type is general like restaurant, decide a random sub-category (e.g., different food types) for the store first. The main overhead signage prominently features a creative, non-infringing fake brand name. If the store type is a public facility, just use the business type as the brand name instead of a fake brand name. Integrated into the main sign and window decals are easily recognizable, large icons **relevant to the business type** (e.g., a stylized cupcake and whisk for a cafe; a retro hanger and boot for a vintage clothing store; a bold red cross and heart for a clinic). The font style should match the business theme.

Materials & Details: The facade is constructed from photorealistic, high-quality textures. The primary materials should match the business theme. Add realistic details like mortar lines, wood grain, or slight weathering appropriate for the materials.

Choose one of the following style and materials according to the business theme: classic: weathered red brick with dark wood trim; industrial: polished concrete with brushed metal panels; rustic: cream stucco with distressed timber beams; modern: glossy white panels with seamless glass; art_deco: black marble with geometric gold brass patterns; japanese: light cypress wood slats with stone base; victorian: ornate painted wood with decorative molding; cyberpunk: dark metal plating with neon signage strips; mediterranean: rough white plaster with terracotta tiles; scandinavian: vertical pale wood cladding with large windows; mid_century: warm teak wood with stone accents; retro: chrome plating with red enamel panels; gothic: dark grey stone with pointed arches; green: living plant wall with cedar framing; budget: beige ceramic tiles with plain aluminum window frames; utility: split-face concrete block with heavy steel doors

Lighting & Glass Constraints: The entire scene is lit with completely flat, neutral, even light (similar to an ambient occlusion pass) to ensure there are no harsh shadows or directional highlights. All windows and glass doors must be fully transparent, revealing a well lit lively interior space behind them. The windows must be completely NON-REFLECTIVE: there should be no reflections of an outside environment on the glass.
"""

prompt_depth = \
"""A purely mathematical, flat-shaded z-depth map of a {place_type} storefront. Darker gray represent higher depth values.

Style: Technical diagram, flat vector art style. UNLIT. No lighting, no shading, no ambient occlusion.

Detail: The depth map should be as detailed as geometry of objects in the RGB image.

Coloring Rules (The "Block" Look):
- Quantized Depth: Use distinct, solid flat blocks of gray color for different depth planes.
- Background: Pure white (RGB 255, 255, 255).
- Windows/Doors: Rendered as SOLID opaque planes. They must be a single flat color code, flush with the frame. Do not render depth for interior objects.
- Order of depth: sign (white) <= wall near sign <= wall near window <= window <= door (black).
- Font and icon: IGNORE any font and icon on the sign. They must have the same depth value as the sign so it's invisible in depth image.

Visual Definition: Sharp, pixel-perfect hard edges. No gradients, no soft shadows, no dirt, no noise. The image should look like a posterized segmentation map, not a 3D render.
"""

place_type_list = []
for place_type, place_number in place_list.items():
    store_front_number = max(place_number//10, 1)
    print(f"Generating {store_front_number} {place_type}")
    for i in range(store_front_number):
        place_type_list.append((place_type, i))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Generating 10 restaurant
Generating 4 fast_food
Generating 2 cafe
Generating 1 bank
Generating 1 bicycle_rental
Generating 1 pub
Generating 1 parking_entrance
Generating 1 clinic
Generating 1 pharmacy
Generating 1 atm
Generating 1 university
Generating 1 bureau_de_change
Generating 1 post_office
Generating 1 theatre
Generating 1 place_of_worship
Generating 1 community_centre
Generating 1 food_court
Generating 1 car_rental
Generating 1 toilets


In [17]:
import os
import threading
from PIL import Image

data_folders = []

notes = {
    "restaurant_7": "Generate a chinese sichuan restaurant",
    "restaurant_9": "Generate a chinese hot pot restaurant",
}

def generate_and_save_storefront(place_type, i):
    place_id = f"{place_type}_{i}"
    folder_path = f"data/{place_id}"
    rgb_path = f"{folder_path}/rgb.png"
    depth_path = f"{folder_path}/depth.png"
    data_folders.append(folder_path)
    os.makedirs(folder_path, exist_ok=True)
    if os.path.exists(rgb_path):
        rgb_image = Image.open(rgb_path)
    else:
        print(f"Generating rgb for {place_type} {i}")
        prompt = prompt_rgb.format(place_type=place_type)
        if place_id in notes:
            prompt = prompt + "\n" + notes[place_id]
        rgb_image = generate_image(prompt)
        rgb_image.save(f"{folder_path}/rgb.png")
    if os.path.exists(depth_path):
        depth_image = Image.open(depth_path)
    else:
        print(f"Generating depth for {place_type} {i}")
        prompt = prompt_depth.format(place_type=place_type)
        depth_image = generate_image([prompt, rgb_image])
        depth_image.save(f"{folder_path}/depth.png")

import queue
threads = []
place_type_queue = queue.Queue()
for place_type, i in place_type_list:
    place_type_queue.put((place_type, i))
while not place_type_queue.empty():
    place_type, i = place_type_queue.get()
    t = threading.Thread(target=generate_and_save_storefront, args=(place_type, i))
    t.start()
    threads.append(t)

for t in threads:
    t.join()


Generating depth for restaurant 0


In [12]:
generate_html(data_folders, "data/storefronts.html")